In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import numpy as np

# Given logits and ground truth
logits = np.array([-2.0, -0.9, -0.3, 0.0, 0.8, 2.2])
truth = np.array([0, 0, 1, 1, 1, 0])

# Convert logits to probabilities
probs = 1 / (1 + np.exp(-logits))

# Q1(a): Logit corresponding to T = 0.3
T = 0.3
logit_T03 = np.log(T / (1 - T))

# Q1(b): Predictions at T = 0.5
pred_strict = (probs > 0.5).astype(int)
pred_equal = (probs >= 0.5).astype(int)

n_strict = pred_strict.sum()
n_equal = pred_equal.sum()

# Q1(c): Dice function
def dice_score(pred, target):
    intersection = np.sum(pred * target)
    return (2 * intersection) / (pred.sum() + target.sum())

pred_T03 = (probs > 0.3).astype(int)
pred_T05 = (probs > 0.5).astype(int)

dice_T03 = dice_score(pred_T03, truth)
dice_T05 = dice_score(pred_T05, truth)

print(f"Logit for T=0.3: {logit_T03:.4f}")
print("Strict predictions at T=0.5:", n_strict)
print("Greater/equal predictions at T=0.5:", n_equal)
print(f"Dice at T=0.3: {dice_T03:.4f}")
print(f"Dice at T=0.5: {dice_T05:.4f}")

Logit for T=0.3: -0.8473
Strict predictions at T=0.5: 2
Greater/equal predictions at T=0.5: 3
Dice at T=0.3: 0.8571
Dice at T=0.5: 0.4000


In [2]:

configs = {
    "X": (0.62, 0.9950, 0.40),
    "Y": (0.58, 0.9990, 0.85),
    "Z": (0.45, 0.9998, 0.97)
}

scores = {}

for name, (fg, fp, empty) in configs.items():
    score = 0.70 * fg + 0.20 * fp + 0.10 * empty
    scores[name] = score
    print(f"Config {name}: {score:.4f}")

best = max(scores, key=scores.get)
print("Best configuration:", best)


Config X: 0.6730
Config Y: 0.6908
Config Z: 0.6120
Best configuration: Y


In [3]:

import numpy as np
import cv2

mask = np.array([
    [1, 1, 0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 1, 0],
    [1, 0, 0, 0, 0, 1, 1, 1]
], dtype=np.uint8)

# 8-connectivity
n8, labels8, stats8, _ = cv2.connectedComponentsWithStats(
    mask, connectivity=8
)

# 4-connectivity
n4, labels4, stats4, _ = cv2.connectedComponentsWithStats(
    mask, connectivity=4
)

areas8 = stats8[1:, cv2.CC_STAT_AREA]

# Remove components with area < 5
pixels_N5 = areas8[areas8 >= 5].sum()

# Remove components with area < 6
pixels_N6 = areas8[areas8 >= 6].sum()

print("8-connected components:", n8 - 1)
print("4-connected components:", n4 - 1)
print("Foreground pixels remaining at N=5:", pixels_N5)
print("Foreground pixels remaining at N=6:", pixels_N6)
print("8-connected component areas:", areas8)


8-connected components: 4
4-connected components: 6
Foreground pixels remaining at N=5: 11
Foreground pixels remaining at N=6: 6
8-connected component areas: [6 1 1 5]


In [4]:

# Given validation data
positive_pairs = 20
empty_pairs = 80

# Positive pair Dice scores
dice_before = [0.70] * 12 + [0.45] * 8

# After filtering, one 0.45-Dice pair becomes empty
dice_after = [0.70] * 12 + [0.45] * 7 + [0.0]

# Empty-pair predicted pixels before filtering
empty_pixels_before = [0] * 50 + [20] * 20 + [2000] * 10

# After N=50 filtering, 20-pixel speckles are removed
empty_pixels_after = [0] * 70 + [2000] * 10

def official_score(dices, empty_pixels):
    fg_dice = np.mean(dices)
    fp_score = 1 - np.mean(empty_pixels) / 102400
    empty_accuracy = np.mean(np.array(empty_pixels) == 0)

    score = (
        0.70 * fg_dice
        + 0.20 * fp_score
        + 0.10 * empty_accuracy
    )

    return fg_dice, fp_score, empty_accuracy, score

fg_before, fp_before, ea_before, score_before = official_score(
    dice_before, empty_pixels_before
)

fg_after, fp_after, ea_after, score_after = official_score(
    dice_after, empty_pixels_after
)

print(f"Foreground Dice after: {fg_after:.4f}")
print(f"False Positive Score after: {fp_after:.4f}")
print(f"Empty Accuracy after: {ea_after:.4f}")
print(f"Official Score before: {score_before:.4f}")
print(f"Official Score after: {score_after:.4f}")


Foreground Dice after: 0.5775
False Positive Score after: 0.9976
Empty Accuracy after: 0.8750
Official Score before: 0.6820
Official Score after: 0.6913


In [8]:
import pandas as pd

# Find existing DataFrames in the notebook
for name, obj in globals().copy().items():
    if isinstance(obj, pd.DataFrame):
        print(f"\nDataFrame: {name}")
        print("Shape:", obj.shape)
        print("Columns:", list(obj.columns))

In [9]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/competitions/industrial-surface-defect-detection/public/sample_submission.csv
/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv


In [10]:
import pandas as pd

TRAIN_CSV = "/kaggle/input/competitions/industrial-surface-defect-detection/public/train.csv"

df = pd.read_csv(TRAIN_CSV)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

Shape: (45572, 5)

Columns:
['image_id', 'defect_type', 'mask_rle', 'width', 'height']

First 5 rows:


,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128



Data types:
image_id       object
defect_type    object
mask_rle       object
width           int64
height          int64
dtype: object


In [11]:

import numpy as np
import pandas as pd
import cv2
from collections import Counter
from tqdm.auto import tqdm

# RLE decoder: column-major (Fortran order)
def decode_rle(rle, height=128, width=800):
    mask = np.zeros(height * width, dtype=np.uint8)

    if pd.isna(rle) or str(rle).strip() == "":
        return mask.reshape((height, width), order="F")

    numbers = np.asarray(rle.split(), dtype=np.int64)
    starts = numbers[0::2] - 1
    lengths = numbers[1::2]

    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1

    return mask.reshape((height, width), order="F")


# Analyze every positive image-class pair
wiped_total = 0
wiped_by_class = Counter()
positive_by_class = Counter()

positive_rows = df[df["mask_rle"].notna()]

for row in tqdm(positive_rows.itertuples(index=False),
                total=len(positive_rows),
                desc="Analyzing positive masks"):

    mask = decode_rle(row.mask_rle, row.height, row.width)

    positive_by_class[row.defect_type] += 1

    n_components, labels, stats, centroids = (
        cv2.connectedComponentsWithStats(
            mask, connectivity=8
        )
    )

    component_areas = stats[1:, cv2.CC_STAT_AREA]

    # Pair is wiped out if every component has area < 100
    if len(component_areas) == 0 or component_areas.max() < 100:
        wiped_total += 1
        wiped_by_class[row.defect_type] += 1

print("\nQ5 RESULTS")
print("Total positive pairs wiped out:", wiped_total)
print("Wiped pairs by class:", dict(wiped_by_class))
print("Positive pairs by class:", dict(positive_by_class))

print("\nClass losing the most pairs:",
      wiped_by_class.most_common(1)[0][0])


Analyzing positive masks:   0%|          | 0/7201 [00:00<?, ?it/s]


Q5 RESULTS
Total positive pairs wiped out: 48
Wiped pairs by class: {'defect_c': 24, 'defect_a': 22, 'defect_d': 1, 'defect_b': 1}
Positive pairs by class: {'defect_a': 1081, 'defect_c': 4100, 'defect_d': 1707, 'defect_b': 313}

Class losing the most pairs: defect_c


In [12]:

import numpy as np

original = np.array([
    [0.9, 0.6],
    [0.4, 0.2]
])

horizontal = np.array([
    [0.5, 0.8],
    [0.3, 0.4]
])

vertical = np.array([
    [0.1, 0.5],
    [0.7, 0.9]
])

# Invert the flips before averaging
horizontal_back = np.fliplr(horizontal)
vertical_back = np.flipud(vertical)

# Average all three predictions
average = (original + horizontal_back + vertical_back) / 3

# Strict threshold
correct_predictions = (average > 0.5).sum()

# Buggy pipeline: no inverse flips
buggy_average = (original + horizontal + vertical) / 3
buggy_predictions = (buggy_average > 0.5).sum()

print("Horizontal flip dims:", (-1,))
print("Vertical flip dims:", (-2,))
print("\nCorrect averaged predictions:")
print(average)
print("Top-right probability:", round(average[0, 1], 4))
print("Correct positive pixels:", correct_predictions)
print("Buggy positive pixels:", buggy_predictions)


Horizontal flip dims: (-1,)
Vertical flip dims: (-2,)

Correct averaged predictions:
[[0.8        0.66666667]
 [0.3        0.33333333]]
Top-right probability: 0.6667
Correct positive pixels: 2
Buggy positive pixels: 1


In [13]:

import numpy as np

# Rows = models, columns = pixels
predictions = np.array([
    [0.90, 0.55, 0.80, 0.10, 0.60],  # Model A
    [0.40, 0.55, 0.70, 0.20, 0.45],  # Model B
    [0.45, 0.20, 0.90, 0.30, 0.55]   # Model C
])

# Ground truth: positive at p1, p3, p5
truth = np.array([1, 0, 1, 0, 1])

# Q7.1 Soft averaging
soft_probs = predictions.mean(axis=0)
soft_pred = (soft_probs > 0.5).astype(int)

def dice(pred, target):
    return 2 * np.sum(pred * target) / (pred.sum() + target.sum())

dice_soft = dice(soft_pred, truth)

# Q7.2 Hard voting
individual_preds = (predictions > 0.5).astype(int)
votes = individual_preds.sum(axis=0)
vote_pred = (votes >= 2).astype(int)

dice_vote = dice(vote_pred, truth)

# Q7.3 Weighted ensemble at p1
weights = np.array([0.5, 0.3, 0.2])
weighted_p1 = np.sum(predictions[:, 0] * weights)

# Q7.4 Probability-space vs logit-space averaging
logits = np.array([1.0, -3.0])

probabilities = 1 / (1 + np.exp(-logits))
prob_avg = probabilities.mean()

logit_avg = logits.mean()
logit_avg_prob = 1 / (1 + np.exp(-logit_avg))

print("Soft ensemble probabilities:", soft_probs)
print("Soft ensemble Dice:", dice_soft)
print("Hard voting Dice:", dice_vote)
print("Weighted ensemble p1:", weighted_p1)
print("Probability average:", round(prob_avg, 4))
print("Logit average probability:", round(logit_avg_prob, 4))
print("Method positive at T=0.30:",
      "prob_avg" if prob_avg > 0.30 else "logit_avg")


Soft ensemble probabilities: [0.58333333 0.43333333 0.8        0.2        0.53333333]
Soft ensemble Dice: 1.0
Hard voting Dice: 0.6666666666666666
Weighted ensemble p1: 0.66
Probability average: 0.3892
Logit average probability: 0.2689
Method positive at T=0.30: prob_avg


In [14]:

# Q8: Memory and inference-time calculations

images = 5000
classes = 4
height = 128
width = 800

# One probability map set for the full test set
total_values = images * classes * height * width

# Memory in decimal GB
gb_fp32 = total_values * 4 / 1e9
gb_fp16 = total_values * 2 / 1e9
gb_uint8 = total_values * 1 / 1e9

# Naive memory: retain every model-view prediction
gb_A_naive = gb_fp32 * 3 * 3
gb_B_naive = gb_fp32 * 2 * 2

# One image at a time: running accumulator for all classes
accumulator_gb = classes * height * width * 4 / 1e9

# Forward passes and model time
passes_A = images * 3 * 3
passes_B = images * 2 * 2

time_A_min = passes_A * 0.012 / 60
time_B_min = passes_B * 0.012 / 60

# Time available after loading, RLE and I/O
available_minutes = 30 - 8

# Maximum whole models
one_model_3view = images * 3 * 0.012 / 60
one_model_2view = images * 2 * 0.012 / 60

max_models_3view = int(available_minutes // one_model_3view)
max_models_2view = int(available_minutes // one_model_2view)

print("Q8(a): Memory for one full prediction set")
print(f"FP32: {gb_fp32:.3f} GB")
print(f"FP16: {gb_fp16:.3f} GB")
print(f"uint8: {gb_uint8:.3f} GB")

print("\nQ8(b): Naive memory")
print(f"Config A: {gb_A_naive:.3f} GB")
print(f"Config B: {gb_B_naive:.3f} GB")
print("Config A fits 16 GB:", gb_A_naive <= 16)
print("Config B fits 16 GB:", gb_B_naive <= 16)
print(f"Running accumulator: {accumulator_gb:.6f} GB")

print("\nQ8(c): Forward passes and pure model time")
print("Config A passes:", passes_A)
print(f"Config A time: {time_A_min:.2f} minutes")
print("Config B passes:", passes_B)
print(f"Config B time: {time_B_min:.2f} minutes")

print("\nQ8(d): Maximum whole models")
print("Available time:", available_minutes, "minutes")
print("3-view TTA:", max_models_3view)
print("2-view TTA:", max_models_2view)

Q8(a): Memory for one full prediction set
FP32: 8.192 GB
FP16: 4.096 GB
uint8: 2.048 GB

Q8(b): Naive memory
Config A: 73.728 GB
Config B: 32.768 GB
Config A fits 16 GB: False
Config B fits 16 GB: False
Running accumulator: 0.001638 GB

Q8(c): Forward passes and pure model time
Config A passes: 45000
Config A time: 9.00 minutes
Config B passes: 20000
Config B time: 4.00 minutes

Q8(d): Maximum whole models
Available time: 22 minutes
3-view TTA: 7
2-view TTA: 11
